# Scaling Inference Compute with Search and Self-Consistency

Training-time scaling — more parameters, more data, more compute — has a well-understood empirical law (Chinchilla). Inference-time scaling is newer and less charted: instead of a fixed forward pass, you spend more compute at inference to get better answers. The key insight is that for hard reasoning tasks, a model's best answer under many attempts is far better than its first answer. This notebook covers the spectrum of inference-time scaling techniques: from simple **best-of-N sampling** to **beam search**, **self-consistency**, **process reward models**, and **Monte Carlo Tree Search** — all of which trade more inference compute for higher accuracy.

## The Inference-Time Compute Tradeoff

Given a fixed model, we can improve output quality by spending more compute at inference time. The quality improvement depends on:

- **What we sample:** more diverse exploration vs. concentrated high-quality paths.
- **How we score:** outcome rewards (final answer correct?) vs. process rewards (each reasoning step correct?).
- **How we search:** independent samples, beam search, tree search.

The fundamental question: for a fixed compute budget, is it better to train a larger model for one forward pass, or train a smaller model and use inference-time scaling? Recent work on reasoning models (DeepSeek-R1, OpenAI o1) shows that for math and coding tasks, the answer increasingly favors inference-time scaling.

## Best-of-N Sampling

The simplest inference-time scaling strategy: sample $N$ independent responses from the model, score each with a reward model or outcome verifier, and return the highest-scoring one.

$$\text{best-of-N} = \arg\max_{y_i} r(x, y_i), \quad y_i \sim \pi(\cdot \mid x).$$

For pass@1 accuracy $p$, best-of-N achieves expected accuracy:

$$P(\text{at least one correct among } N) = 1 - (1 - p)^N.$$

For $p = 0.3$ and $N = 10$: $1 - 0.7^{10} \approx 0.97$. The model goes from 30% to 97% accuracy with 10× the inference compute — a dramatic improvement. The cost: you need a reliable way to identify the correct answer (a verifier or reward model).

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np


def best_of_n(
    model,
    reward_model,
    tokenizer,
    prompt: str,
    n: int = 8,
    max_new_tokens: int = 256,
    temperature: float = 0.8,
    device: torch.device = None,
) -> tuple[str, list[float]]:
    """Sample N responses and return the highest-reward one.

    Args:
        model: language model for generation.
        reward_model: scalar reward model; takes tokenized (prompt + response).
        tokenizer: tokenizer with encode/decode.
        prompt: input prompt string.
        n: number of candidate responses to sample.
        max_new_tokens: max generation length per candidate.
        temperature: sampling temperature.
        device: compute device.

    Returns:
        best_response: the response string with the highest reward.
        rewards: list of rewards for all n candidates.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )

    candidates = []
    with torch.no_grad():
        for _ in range(n):
            output = model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
            )[0, prompt_ids.size(1):]
            candidates.append(output.cpu())

    # Score all candidates
    rewards = []
    reward_model.eval()
    with torch.no_grad():
        for resp_ids in candidates:
            full_ids = torch.cat([prompt_ids[0].cpu(), resp_ids]).unsqueeze(0).to(device)
            reward = reward_model(full_ids).squeeze().item()
            rewards.append(reward)

    best_idx = int(np.argmax(rewards))
    best_response = tokenizer.decode(candidates[best_idx].tolist())
    return best_response, rewards


def pass_at_k_curve(base_accuracy: float, n_values: list[int]) -> list[float]:
    """Theoretical pass@1 accuracy as a function of best-of-N.

    Args:
        base_accuracy: single-sample accuracy p.
        n_values: list of N values to evaluate.

    Returns:
        List of pass@1 accuracies for each N.
    """
    return [1 - (1 - base_accuracy) ** n for n in n_values]

## Beam Search

Best-of-N generates $N$ independent samples. Beam search maintains $B$ **beams** — partial sequences — and at each step extends each beam, scores the extensions, and keeps the top-$B$ partial sequences. Unlike independent sampling, beam search concentrates exploration on the most promising prefixes.

Beam search maximizes:

$$\hat{y} = \arg\max_{y} \sum_{t=1}^{|y|} \log p(y_t \mid x, y_{<t}) / |y|^\alpha,$$

where $\alpha$ is a length penalty to prevent shorter sequences from being favored.

For language model generation, beam search is [mode-seeking]{.mark}: it finds the highest-probability completion. This is often bad for open-ended generation (produces repetitive, generic text) but good for constrained tasks with clear correct answers (translation, math, code).

In [ ]:
def beam_search(
    model,
    tokenizer,
    prompt: str,
    beam_width: int = 4,
    max_new_tokens: int = 128,
    length_penalty: float = 0.6,
    device: torch.device = None,
) -> list[tuple[str, float]]:
    """Beam search for language model generation.

    Args:
        model: language model.
        tokenizer: tokenizer with encode/decode and eos_id attribute.
        prompt: input prompt string.
        beam_width: number of beams to maintain.
        max_new_tokens: maximum generation length.
        length_penalty: exponent for length normalization (0 = no penalty).
        device: compute device.

    Returns:
        List of (text, score) tuples sorted by descending score,
        one per completed beam.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    prompt_ids = tokenizer.encode(prompt)
    prompt_len = len(prompt_ids)

    # Beams: list of (token_ids_list, cumulative_log_prob)
    beams = [(prompt_ids[:], 0.0)]
    completed = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            if not beams:
                break

            all_candidates = []
            for ids, score in beams:
                input_ids = torch.tensor([ids], dtype=torch.long, device=device)
                logits = model(input_ids)[0, -1, :]        # (V,)
                log_probs = F.log_softmax(logits, dim=-1)   # (V,)

                # Expand: consider top beam_width extensions
                topk_log_probs, topk_ids = log_probs.topk(beam_width)
                for log_p, next_id in zip(topk_log_probs.tolist(), topk_ids.tolist()):
                    new_ids = ids + [next_id]
                    new_score = score + log_p              # <1>
                    all_candidates.append((new_ids, new_score))

            # Keep top beam_width by length-normalized score
            def normalized_score(item):
                ids, score = item
                response_len = len(ids) - prompt_len
                return score / max(response_len, 1) ** length_penalty  # <2>

            all_candidates.sort(key=normalized_score, reverse=True)
            beams = []
            for ids, score in all_candidates[:beam_width * 2]:
                if ids[-1] == tokenizer.eos_id or len(ids) - prompt_len >= max_new_tokens:
                    completed.append((ids, score))
                else:
                    if len(beams) < beam_width:
                        beams.append((ids, score))

    # Add any remaining active beams to completed
    completed.extend(beams)
    completed.sort(key=normalized_score, reverse=True)

    results = []
    for ids, score in completed[:beam_width]:
        text = tokenizer.decode(ids[prompt_len:])
        response_len = len(ids) - prompt_len
        norm_score = score / max(response_len, 1) ** length_penalty
        results.append((text, norm_score))

    return results

1. Cumulative log-probability: the score of a beam is the sum of log-probabilities of all tokens generated so far.
2. Length normalization: divide by `response_len^alpha` to prevent the model from preferring short completions (which accumulate less negative log-probability).

## Self-Consistency

Self-consistency [@wang2023selfconsistency] is an inference-time technique for problems with verifiable answers: sample $N$ reasoning chains and return the answer that appears most frequently. Unlike best-of-N with a reward model, self-consistency requires no external scorer — it relies on the statistical regularity that correct reasoning paths are more likely to converge on the same answer.

For math problems: generate $N$ solutions, extract the final numerical answer from each, and return the majority vote.

This works because:
- Incorrect solutions are diverse (many ways to be wrong)
- Correct solutions are concentrated (few ways to arrive at the right answer)

Self-consistency is [most effective when the model has moderate accuracy]{.mark} (20%–80% per sample). At very high accuracy, best-of-1 is sufficient. At very low accuracy, self-consistency also fails because there is no correct answer to converge on.

In [ ]:
import re
from collections import Counter


def extract_final_answer(text: str) -> str | None:
    """Extract a numeric or symbolic final answer from a reasoning chain.

    Looks for patterns like 'the answer is X', 'X is the answer',
    '= X', or the last standalone number in the text.

    Args:
        text: model-generated reasoning chain string.

    Returns:
        Extracted answer string, or None if no answer found.
    """
    # Try explicit answer patterns
    patterns = [
        r"(?:the answer is|answer:|therefore,?|so,?)\s*([\-\d\.]+)",
        r"=\s*([\-\d\.]+)\s*$",
        r"([\-\d\.]+)\s*$",
    ]
    for pattern in patterns:
        match = re.search(pattern, text.lower().strip())
        if match:
            return match.group(1).strip()
    return None


def self_consistency(
    model,
    tokenizer,
    prompt: str,
    n: int = 16,
    max_new_tokens: int = 512,
    temperature: float = 0.8,
    extract_fn=None,
    device: torch.device = None,
) -> tuple[str | None, dict]:
    """Self-consistency decoding: majority vote over N sampled solutions.

    Args:
        model: language model.
        tokenizer: tokenizer.
        prompt: input prompt.
        n: number of solutions to sample.
        max_new_tokens: max generation length per solution.
        temperature: sampling temperature.
        extract_fn: callable(text) -> answer_str; defaults to extract_final_answer.
        device: compute device.

    Returns:
        majority_answer: the most common extracted answer, or None.
        info: dict with 'vote_counts', 'all_answers', 'all_solutions'.
    """
    if extract_fn is None:
        extract_fn = extract_final_answer
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )

    all_solutions = []
    all_answers = []

    with torch.no_grad():
        for _ in range(n):
            output = model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
            )[0, prompt_ids.size(1):]
            solution = tokenizer.decode(output.tolist())
            all_solutions.append(solution)
            all_answers.append(extract_fn(solution))

    valid_answers = [a for a in all_answers if a is not None]
    if not valid_answers:
        return None, {"vote_counts": {}, "all_answers": all_answers, "all_solutions": all_solutions}

    vote_counts = Counter(valid_answers)
    majority_answer = vote_counts.most_common(1)[0][0]

    return majority_answer, {
        "vote_counts": dict(vote_counts),
        "all_answers": all_answers,
        "all_solutions": all_solutions,
    }

## Process Reward Models

Outcome reward models score the final answer. **Process reward models (PRMs)** score each individual reasoning step, providing a denser signal that can guide search more precisely.

A PRM takes a partial reasoning chain as input and predicts the probability that the current step is correct, given the prompt and all previous steps:

$$r_\text{PRM}(x, y_{1:t}) \in [0, 1].$$

PRMs are trained on step-level annotations: human (or model-generated) labels marking each step as correct or incorrect. This is more expensive to collect than outcome labels but provides much richer training signal.

In practice, a PRM can be implemented as a language model with a scalar head that is fine-tuned on step-level labels — essentially a per-step reward model rather than a per-response one.

In [ ]:
import torch.nn as nn


class ProcessRewardModel(nn.Module):
    """Reward model that scores individual reasoning steps.

    Takes a partial reasoning chain (prompt + steps so far) and
    predicts a scalar reward for the most recent step.

    Args:
        base_model: language model backbone.
        d_model: hidden dimension of the backbone.
    """

    def __init__(self, base_model: nn.Module, d_model: int) -> None:
        super().__init__()
        self.base_model = base_model
        self.reward_head = nn.Linear(d_model, 1)           # <1>

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Score a partial reasoning chain.

        Args:
            input_ids: shape (B, T) — tokenized prompt + steps so far.

        Returns:
            reward: shape (B,) — scalar score for the last step.
        """
        hidden = self.base_model(input_ids, return_hidden=True)  # (B, T, d_model)
        last_hidden = hidden[:, -1, :]                            # (B, d_model)  # <2>
        return self.reward_head(last_hidden).squeeze(-1)          # (B,)


def score_reasoning_steps(
    prm: ProcessRewardModel,
    tokenizer,
    prompt: str,
    steps: list[str],
    device: torch.device = None,
) -> list[float]:
    """Score each step in a reasoning chain using the PRM.

    Args:
        prm: process reward model.
        tokenizer: tokenizer.
        prompt: the original question prompt.
        steps: list of reasoning step strings.
        device: compute device.

    Returns:
        List of step scores, one per step.
    """
    if device is None:
        device = next(prm.parameters()).device

    prm.eval()
    scores = []
    accumulated = prompt

    with torch.no_grad():
        for step in steps:
            accumulated = accumulated + "\n" + step
            input_ids = torch.tensor(
                [tokenizer.encode(accumulated)], dtype=torch.long, device=device
            )
            score = prm(input_ids).item()
            scores.append(score)

    return scores

1. A single linear projection from the last hidden state to a scalar reward — the same architecture as the outcome reward model in [NB09](/courses/llm/09-preference-optimization.html), but trained on step-level labels.
2. The last token's hidden state summarizes the full context — used as the representation for the current step.

## Tree Search

Best-of-N and self-consistency generate complete solutions independently. Tree search integrates generation and evaluation more tightly: at each step, a partial solution is evaluated and the search explores the most promising branches, abandoning unlikely paths early.

**Monte Carlo Tree Search (MCTS)** adapted to language generation maintains a tree where each node is a partial reasoning chain. The four MCTS operations:

1. **Select:** traverse the tree from the root using the UCB1 formula to balance exploration and exploitation.
2. **Expand:** generate one or more next steps from the selected node.
3. **Simulate:** run the partial chain to completion (or use the PRM score directly as a value estimate).
4. **Backpropagate:** update value estimates back up the tree.

**Simpler variant: Lookahead search.** At each step, generate $k$ candidate next tokens/sentences, score each with the PRM, pick the best, and repeat. This is a greedy tree search with depth 1 lookahead — much cheaper than full MCTS.

In [ ]:
from dataclasses import dataclass, field as datafield


@dataclass
class SearchNode:
    """A node in the reasoning search tree.

    Attributes:
        steps: list of reasoning steps from root to this node.
        score: cumulative PRM score.
        visit_count: number of times this node was visited.
        children: list of child nodes.
    """
    steps: list[str]
    score: float = 0.0
    visit_count: int = 0
    children: list["SearchNode"] = datafield(default_factory=list)

    @property
    def depth(self) -> int:
        return len(self.steps)

    def ucb1(self, parent_visits: int, c: float = 1.41) -> float:
        """UCB1 score for tree traversal.

        Args:
            parent_visits: number of visits to the parent node.
            c: exploration constant (sqrt(2) by default).

        Returns:
            UCB1 value; higher = more worth exploring.
        """
        if self.visit_count == 0:
            return float("inf")
        exploitation = self.score / self.visit_count
        exploration = c * (np.log(parent_visits) / self.visit_count) ** 0.5
        return exploitation + exploration


def lookahead_search(
    model,
    prm: ProcessRewardModel,
    tokenizer,
    prompt: str,
    n_steps: int = 8,
    candidates_per_step: int = 4,
    max_tokens_per_step: int = 128,
    temperature: float = 0.8,
    device: torch.device = None,
) -> list[str]:
    """Greedy lookahead search: at each step generate k candidates,
    score with PRM, and select the best.

    Args:
        model: language model.
        prm: process reward model.
        tokenizer: tokenizer.
        prompt: input prompt.
        n_steps: number of reasoning steps to generate.
        candidates_per_step: branching factor at each step.
        max_tokens_per_step: max tokens per candidate step.
        temperature: sampling temperature.
        device: compute device.

    Returns:
        List of selected reasoning steps (best path).
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    prm.eval()
    accumulated_steps = []
    context = prompt

    with torch.no_grad():
        for step_idx in range(n_steps):
            # Generate candidates_per_step candidate next steps
            candidate_texts = []
            context_ids = torch.tensor(
                [tokenizer.encode(context)], dtype=torch.long, device=device
            )
            for _ in range(candidates_per_step):
                output = model.generate(
                    context_ids,
                    max_new_tokens=max_tokens_per_step,
                    temperature=temperature,
                    do_sample=True,
                    eos_token_id=tokenizer.newline_id if hasattr(tokenizer, "newline_id") else None,
                )[0, context_ids.size(1):]
                candidate_texts.append(tokenizer.decode(output.tolist()))

            # Score each candidate with PRM
            scores = []
            for cand in candidate_texts:
                cand_steps = accumulated_steps + [cand]
                step_scores = score_reasoning_steps(prm, tokenizer, prompt, cand_steps, device)
                scores.append(step_scores[-1])             # <1>

            best_idx = int(np.argmax(scores))
            best_step = candidate_texts[best_idx]
            accumulated_steps.append(best_step)
            context = context + "\n" + best_step

    return accumulated_steps

Annotation:

1. Score only the last step — the PRM scores the most recent step given all previous context. We select the candidate whose last step receives the highest PRM score.

## Compute-Quality Tradeoffs

All inference-time scaling methods trade compute for quality. The relevant axis is **FLOPs per correct answer**, not just FLOPs per token. A method that uses 10× the compute but achieves 97% accuracy (vs 30% for standard decoding) is a 3× improvement in FLOPs per correct answer.

Empirical findings from the reasoning model literature:

- **Best-of-N with a strong verifier** scales well to $N \sim 100$–$1000$ for math problems. The scaling is approximately $\log N$ in accuracy.
- **Self-consistency** plateaus earlier than best-of-N with a reward model — the majority vote saturates when many incorrect paths converge on the same wrong answer.
- **Tree search with PRM** dominates best-of-N at equal compute for multi-step reasoning tasks, because it avoids wasting compute on paths that are already known to be poor.
- **Chain-of-thought length** matters: longer chains of thought (more inference tokens) improve reasoning accuracy up to a point, then plateau or degrade.

Quantifying the scaling behavior:

In [ ]:
def compute_scaling_curve(
    base_accuracy: float,
    compute_multipliers: list[float],
    method: str = "best_of_n",
) -> list[float]:
    """Theoretical accuracy as a function of inference compute multiplier.

    Args:
        base_accuracy: single-sample pass@1 accuracy.
        compute_multipliers: list of compute multipliers (e.g. [1, 2, 4, 8, 16]).
        method: 'best_of_n' or 'self_consistency' (approximate model).

    Returns:
        List of expected accuracies at each compute multiplier.
    """
    accuracies = []
    for multiplier in compute_multipliers:
        n = max(1, int(multiplier))
        if method == "best_of_n":
            # Assumes a perfect verifier: accuracy = 1 - (1-p)^N
            acc = 1 - (1 - base_accuracy) ** n
        elif method == "self_consistency":
            # Self-consistency saturates faster — approximate with sqrt scaling
            # (empirical observation, not a closed-form result)
            acc = min(1.0, base_accuracy + (1 - base_accuracy) * (1 - 1 / n ** 0.5))
        else:
            raise ValueError(f"Unknown method: {method}")
        accuracies.append(acc)
    return accuracies

:::{.callout-note}
The **o1 model** (OpenAI, 2024) and **DeepSeek-R1** demonstrate that inference-time scaling via long chain-of-thought generation can substantially outperform larger models with shorter generation on hard reasoning benchmarks. The key architectural change: training the model with RL (GRPO or similar) to generate extended internal reasoning before outputting a final answer — turning inference-time compute into a first-class optimization target.

:::

## Summary

| Concept | Key detail |
|---|---|
| Best-of-N | Sample $N$ completions, return highest-scoring. Success rate: $1 - (1-p)^N$. |
| Beam search | Expand top-$k$ partial sequences by log-probability; length-normalize to avoid short-sequence bias. |
| Self-consistency | Sample $N$ answers, majority-vote. Works without a reward model. |
| Process reward model (PRM) | Scores each intermediate step, not just the final answer. [Enables step-level search]{.mark}. |
| MCTS | Select → expand → simulate → backpropagate. PRM-guided tree search over reasoning steps. |
| Compute-quality tradeoff | All methods trade inference FLOPs for accuracy; diminishing returns above a method-specific $N$. |

: {tbl-colwidths="[30,70]"}

## Exercises

1. **Best-of-N scaling curve.** For base accuracy $p \in \{0.1, 0.3, 0.5, 0.7\}$, plot expected accuracy vs $N \in \{1, 2, 4, 8, 16, 32, 64\}$ using `compute_scaling_curve`. At what $N$ does the 0.3 base accuracy model match a 0.7 base accuracy model at $N=1$?

2. **Beam search vs sampling.** Run both `beam_search` and top-$k$ sampling on 20 math problems. Compare pass@1 accuracy, response diversity (unique answers), and mean response length. When does beam search outperform sampling?

3. **Self-consistency vote counts.** Run `self_consistency` with $N \in \{4, 8, 16, 32\}$ on 50 arithmetic problems. Plot accuracy vs $N$ and compare to the best-of-N theoretical curve. At what $N$ does self-consistency plateau?

4. **PRM training.** Given a dataset of reasoning chains with step-level labels, train a `ProcessRewardModel` by fine-tuning a small language model backbone. Evaluate step-level precision and recall against human annotations.

5. **Lookahead depth.** Implement a version of `lookahead_search` that looks ahead $d$ steps (generating $k^d$ candidate paths and scoring all completions). Compare accuracy vs FLOPs for depth $d \in \{1, 2, 3\}$. At what depth does the improvement saturate?

6. **Compute-optimal allocation.** Given a fixed compute budget $C$ and two options — (a) a larger model at 1× samples, (b) a smaller model (0.5× FLOPs per sample) with best-of-$N$ — find the crossover point where option (b) becomes better, as a function of base accuracy.

■